# Chicago Crimes - Feature Engineering (Temporal & Spatial)
Input: chicago_crimes_clean.parquet (hasil preprocessing). 

Dimensi fitur: SPASIAL (grid aggregation) + TEMPORAL (cyclical encoding + tren/recency + siklus harian/mingguan).

## 0. Setup

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
warnings.filterwarnings('ignore')
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 60)
BRAND_INDIGO='#3D2B9E'; BRAND_VIOLET='#7B3FE4'; BRAND_MAGENTA='#CE1C8E'; BRAND_PINK='#EC6DB4'; BRAND_LIGHT='#F7B4DA'
BRAND_COLORS=[BRAND_MAGENTA,BRAND_INDIGO,BRAND_VIOLET,BRAND_PINK,BRAND_LIGHT]
ACCENT=BRAND_MAGENTA
BRAND_CMAP=LinearSegmentedColormap.from_list('brand',['#241663',BRAND_INDIGO,BRAND_VIOLET,BRAND_MAGENTA,BRAND_PINK,BRAND_LIGHT])
try: mpl.colormaps.register(BRAND_CMAP)
except (ValueError, AttributeError): pass
plt.rcParams['axes.prop_cycle']=plt.cycler(color=BRAND_COLORS); plt.rcParams['image.cmap']='brand'
plt.rcParams['figure.figsize']=(12,5); plt.rcParams['figure.dpi']=110; plt.rcParams['axes.grid']=True; plt.rcParams['grid.alpha']=0.25
print('pandas', pd.__version__, '| numpy', np.__version__, '| mode CPU')

pandas 2.3.3 | numpy 2.0.2 | mode CPU


## 1. Konfigurasi

In [2]:
DATASET_DIR='/kaggle/input/datasets/fransiskaadin/dataset-chicago-crimes'
OUT_DIR='/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
GRID=0.01                 # ~1.1 km per sel
LAT_MIN,LAT_MAX=41.6,42.1
LON_MIN,LON_MAX=-87.95,-87.5
VIOLENT={'HOMICIDE','CRIMINAL SEXUAL ASSAULT','CRIM SEXUAL ASSAULT','ROBBERY','BATTERY','ASSAULT','KIDNAPPING','INTIMIDATION','HUMAN TRAFFICKING'}
PROPERTY={'THEFT','BURGLARY','MOTOR VEHICLE THEFT','CRIMINAL DAMAGE','ARSON','CRIMINAL TRESPASS'}
LABEL_BASIS=['n_crimes','n_violent','n_property','n_arrest','n_domestic','n_types','arrest_rate','domestic_rate','share_night','share_weekend']
FEATURE_COLS=['glat','glon','gi','gj','year_idx','pmonth','pquarter','pmonth_sin','pmonth_cos',
  'lag_1','lag_3','lag_12','roll3_mean','roll6_mean','roll12_mean','cell_hist_mean','trend_3_12',
  'n_violent_lag1','arrest_rate_lag1','domestic_rate_lag1','neighbor_lag1_mean','share_night_lag1','share_weekend_lag1']
print('GRID', GRID, '| OUT_DIR', OUT_DIR, '| n fitur', len(FEATURE_COLS))

GRID 0.01 | OUT_DIR /kaggle/working | n fitur 23


## 2. Load data bersih

In [3]:
pq=[DATASET_DIR+'/chicago_crimes_clean.parquet','/kaggle/working/chicago_crimes_clean.parquet','chicago_crimes_clean.parquet']
pq+=sorted(glob.glob('/kaggle/input/**/chicago_crimes_clean.parquet', recursive=True))
df=None; src=None
for p in pq:
    if os.path.exists(p):
        try: df=pd.read_parquet(p); src=p; break
        except Exception: pass
if df is None:
    for p in [x.replace('.parquet','.csv') for x in pq]:
        if os.path.exists(p):
            df=pd.read_csv(p, low_memory=False); src=p; break
if df is None:
    raise FileNotFoundError('chicago_crimes_clean tidak ditemukan (parquet/csv).')
print('Loaded:', src, '| shape:', df.shape)
print('Kolom:', list(df.columns)[:30])

Loaded: /kaggle/input/datasets/fransiskaadin/dataset-chicago-crimes/chicago_crimes_clean.parquet | shape: (8534663, 24)
Kolom: ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'primary_type_canon', 'location_desc_clean', 'coord_missing']


## 3. Normalisasi tipe & turunan insiden
Robust terhadap sumber parquet (tipe sudah rapi) maupun CSV (tipe string). Membentuk:
grid index (gi,gj), periode bulanan, dan sinyal waktu insiden (jam malam & akhir pekan).

In [4]:
# Date -> datetime
if not pd.api.types.is_datetime64_any_dtype(df['Date']):
    dt=pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
    if dt.isna().mean()>0.5: dt=pd.to_datetime(df['Date'], errors='coerce')
    df['Date']=dt
# Arrest/Domestic -> bool
def to_bool(s):
    if s.dtype==bool: return s
    return s.astype('str').str.strip().str.lower().isin(['true','1','yes','t'])
for c in ['Arrest','Domestic']:
    if c in df.columns: df[c]=to_bool(df[c])
    else: df[c]=False
# primary_type_canon
if 'primary_type_canon' not in df.columns:
    df['primary_type_canon']=df['Primary Type'].astype('str').str.upper().str.strip()
if 'Description' not in df.columns: df['Description']=''
# koordinat & coord_missing
lat=pd.to_numeric(df['Latitude'], errors='coerce'); lon=pd.to_numeric(df['Longitude'], errors='coerce')
inbbox=lat.between(LAT_MIN,LAT_MAX)&lon.between(LON_MIN,LON_MAX)
base_missing=df['coord_missing'].astype(bool) if 'coord_missing' in df.columns else pd.Series(False, index=df.index)
df['coord_missing']=base_missing | lat.isna() | lon.isna() | (~inbbox)
geo=~df['coord_missing']
# grid index
df['gi']=np.where(geo, np.round(lat/GRID), np.nan)
df['gj']=np.where(geo, np.round(lon/GRID), np.nan)
df['glat']=df['gi']*GRID; df['glon']=df['gj']*GRID
# periode & sinyal waktu
df['period_str']=df['Date'].dt.to_period('M').astype('str')
df['year']=df['Date'].dt.year; df['month']=df['Date'].dt.month
df['hour']=df['Date'].dt.hour; df['dow']=df['Date'].dt.dayofweek
df['is_night']=(((df['hour']>=22)|(df['hour']<5))).astype('int8')
df['is_weekend']=(df['dow']>=5).astype('int8')
df['is_violent']=df['primary_type_canon'].isin(VIOLENT).astype('int8')
df['is_property']=df['primary_type_canon'].isin(PROPERTY).astype('int8')
print('Geocoded:', int(geo.sum()), '(', round(100*geo.mean(),2), '% )')
print('Rentang periode:', df['period_str'].min(), '->', df['period_str'].max())

Geocoded: 8438128 ( 98.87 % )
Rentang periode: 2001-01 -> 2026-04


Dari data bersih, 98,87% baris tergeocode dan rentang periode 2001-01 hingga 2026-04 dinormalkan ke `period_str` bulanan. Di tahap ini dibentuk penanda tingkat-insiden (mis. `is_night`, `is_weekend`, `is_violent`, `is_property`) yang menjadi bahan mentah agregasi. Bulan dipilih sebagai granularitas waktu karena menyeimbangkan sinyal musiman dengan jumlah sampel per sel yang cukup stabil.


## 4. Simpan features_incident (dipakai pseudo-labeling)

In [5]:
inc_cols=['Date','period_str','year','month','hour','dow','primary_type_canon','Primary Type','Description',
  'Location Description','Arrest','Domestic','Latitude','Longitude','coord_missing','gi','gj','glat','glon',
  'is_night','is_weekend','is_violent','is_property']
inc_cols=[c for c in inc_cols if c in df.columns]
features_incident=df[inc_cols].copy()
try:
    features_incident.to_parquet(os.path.join(OUT_DIR,'features_incident.parquet'), index=False)
    print('Tersimpan features_incident.parquet', features_incident.shape)
except Exception as e:
    features_incident.to_csv(os.path.join(OUT_DIR,'features_incident.csv'), index=False)
    print('Fallback CSV features_incident.csv ('+type(e).__name__+')', features_incident.shape)

Tersimpan features_incident.parquet (8534663, 23)


`features_incident` (8,53 jt baris) sengaja disimpan pada level insiden untuk mempertahankan `Primary Type` + `Description` yang dibutuhkan pseudo-labeling untuk menghitung severity. Ini memisahkan tanggung jawab: fitur level-insiden untuk severity, fitur level-sel untuk konteks model.


## 5. Agregasi spasial-temporal ke sel grid x bulan
Unit analisis = sel grid (0.01 deg) x bulan. share_night & share_weekend mempertahankan sinyal
siklus harian & mingguan yang hilang saat agregasi bulanan.

In [6]:
g=df[(~df['coord_missing'])&(df['Date'].notna())]
agg=g.groupby(['gi','gj','period_str'], observed=True).agg(
    n_crimes=('primary_type_canon','size'), n_violent=('is_violent','sum'), n_property=('is_property','sum'),
    n_arrest=('Arrest','sum'), n_domestic=('Domestic','sum'), n_types=('primary_type_canon','nunique'),
    night=('is_night','sum'), weekend=('is_weekend','sum')).reset_index()
agg['arrest_rate']=agg['n_arrest']/agg['n_crimes']
agg['domestic_rate']=agg['n_domestic']/agg['n_crimes']
agg['share_night']=agg['night']/agg['n_crimes']
agg['share_weekend']=agg['weekend']/agg['n_crimes']
print('Sel x bulan teramati:', agg.shape, '| sel unik:', agg[['gi','gj']].drop_duplicates().shape[0])

Sel x bulan teramati: (197608, 15) | sel unik: 746


Koordinat kontinu dibin ke grid 0,01 derajat (~1,1 km) lalu diagregasi per bulan, menghasilkan 197.608 sel-bulan teramati dari 746 sel unik. Grid menjawab pertanyaan spasial task: dua titik berdekatan kini masuk sel yang sama sehingga bisa dibandingkan dan dihitung densitasnya secara bermakna.


## 6. Panel penuh (sel x semua periode) & fitur kalender/siklikal

In [7]:
cells=agg[['gi','gj']].drop_duplicates().assign(k=1)
periods=pd.DataFrame({'period_str':sorted(agg['period_str'].unique())}).assign(k=1)
panel=cells.merge(periods, on='k').drop(columns='k').merge(agg, on=['gi','gj','period_str'], how='left')
cnt=['n_crimes','n_violent','n_property','n_arrest','n_domestic','n_types','night','weekend']
rate=['arrest_rate','domestic_rate','share_night','share_weekend']
panel[cnt]=panel[cnt].fillna(0); panel[rate]=panel[rate].fillna(0)
panel['glat']=panel['gi']*GRID; panel['glon']=panel['gj']*GRID
panel['period_dt']=pd.to_datetime(panel['period_str']+'-01')
panel=panel.sort_values(['gi','gj','period_dt']).reset_index(drop=True)
panel['year']=panel['period_dt'].dt.year; panel['pmonth']=panel['period_dt'].dt.month
panel['year_idx']=panel['year']-int(panel['year'].min())
panel['pquarter']=((panel['pmonth']-1)//3+1).astype(int)
panel['pmonth_sin']=np.sin(2*np.pi*panel['pmonth']/12)
panel['pmonth_cos']=np.cos(2*np.pi*panel['pmonth']/12)
print('Panel penuh:', panel.shape, '=', cells.shape[0], 'sel x', periods.shape[0], 'periode')

Panel penuh: (226784, 24) = 746 sel x 304 periode


Panel diperluas menjadi 746 sel x 304 periode = 226.784 baris, termasuk sel-bulan tanpa kejadian (diisi 0). Tanpa langkah ini, bulan-bulan tenang akan hilang dan risiko akan terlihat lebih tinggi dari seharusnya. Di sini juga dibentuk fitur kalender siklikal `pmonth_sin`/`pmonth_cos` - jawaban langsung atas masalah "Desember dan Januari berdekatan" yang linear tidak bisa tangkap.


## 7. Fitur tren & recency (lag, rolling) - vektorized & kausal
Semua memakai shift(>=1) sehingga hanya bergantung pada masa lalu (bebas leakage).

In [8]:
gp=panel.groupby(['gi','gj'], observed=True)
panel['lag_1']=gp['n_crimes'].shift(1); panel['lag_3']=gp['n_crimes'].shift(3); panel['lag_12']=gp['n_crimes'].shift(12)
panel['roll3_mean']=gp['n_crimes'].transform(lambda s: s.shift(1).rolling(3).mean())
panel['roll6_mean']=gp['n_crimes'].transform(lambda s: s.shift(1).rolling(6).mean())
panel['roll12_mean']=gp['n_crimes'].transform(lambda s: s.shift(1).rolling(12).mean())
panel['cell_hist_mean']=gp['n_crimes'].transform(lambda s: s.shift(1).expanding().mean())
panel['trend_3_12']=panel['roll3_mean']-panel['roll12_mean']
panel['n_violent_lag1']=gp['n_violent'].shift(1)
panel['arrest_rate_lag1']=gp['arrest_rate'].shift(1)
panel['domestic_rate_lag1']=gp['domestic_rate'].shift(1)
panel['share_night_lag1']=gp['share_night'].shift(1)
panel['share_weekend_lag1']=gp['share_weekend'].shift(1)
print('Fitur lag/rolling selesai.')

Fitur lag/rolling selesai.


`lag_1/3/12`, `roll3/6/12_mean`, `cell_hist_mean`, dan `trend_3_12` semuanya dihitung dengan shift ke masa lalu per sel - tidak ada kebocoran masa depan (no leakage). Fitur ini memberi model memori historis: momentum jangka pendek, siklus tahunan (lag_12), dan tren struktural sebuah lokasi.


## 8. Fitur spillover spasial (rata-rata lag tetangga)

In [9]:
lag_series=panel.set_index(['gi','gj','period_str'])['lag_1']
gi=panel['gi'].to_numpy(); gj=panel['gj'].to_numpy(); per=panel['period_str'].to_numpy()
total=np.zeros(len(panel)); count=np.zeros(len(panel))
for di in (-1,0,1):
    for dj in (-1,0,1):
        if di==0 and dj==0: continue
        idx=list(zip(gi+di, gj+dj, per))
        vals=lag_series.reindex(idx).to_numpy()
        m=~np.isnan(vals)
        total=total+np.where(m, vals, 0.0); count=count+m
panel['neighbor_lag1_mean']=np.where(count>0, total/np.where(count>0,count,1.0), np.nan)
print('neighbor_lag1_mean selesai. rata-rata:', round(float(np.nanmean(panel['neighbor_lag1_mean'])),3))

neighbor_lag1_mean selesai. rata-rata: 38.364


`neighbor_lag1_mean` (rata-rata 38,36) merangkum aktivitas kejahatan di sel-sel tetangga pada periode sebelumnya. Fitur ini mengakui bahwa risiko sebuah lokasi tidak berdiri sendiri - kejahatan di blok sebelah relevan. Ini pelengkap eksplisit untuk konsep spatial decay yang diformalkan di pseudo-labeling.


## 9. Rakit & simpan features_gridmonth

In [10]:
keep=['gi','gj','period_str']+LABEL_BASIS+FEATURE_COLS
keep=list(dict.fromkeys(keep))  # buang duplikat gi/gj
fgm=panel[keep].copy()
fgm[FEATURE_COLS]=fgm[FEATURE_COLS].fillna(0)
print('features_gridmonth:', fgm.shape, '| NaN total:', int(fgm.isna().sum().sum()))
print(fgm[['gi','gj','period_str','n_crimes','share_night','share_weekend','lag_1','share_night_lag1','neighbor_lag1_mean']].head(8).to_string(index=False))
try:
    fgm.to_parquet(os.path.join(OUT_DIR,'features_gridmonth.parquet'), index=False)
    print('Tersimpan features_gridmonth.parquet', fgm.shape)
except Exception as e:
    fgm.to_csv(os.path.join(OUT_DIR,'features_gridmonth.csv'), index=False)
    print('Fallback CSV features_gridmonth.csv ('+type(e).__name__+')', fgm.shape)

features_gridmonth: (226784, 34) | NaN total: 0
    gi      gj period_str  n_crimes  share_night  share_weekend  lag_1  share_night_lag1  neighbor_lag1_mean
4164.0 -8762.0    2001-01       0.0          0.0            0.0    0.0               0.0            0.000000
4164.0 -8762.0    2001-02       0.0          0.0            0.0    0.0               0.0           11.666667
4164.0 -8762.0    2001-03       0.0          0.0            0.0    0.0               0.0           10.666667
4164.0 -8762.0    2001-04       0.0          0.0            0.0    0.0               0.0           17.000000
4164.0 -8762.0    2001-05       0.0          0.0            0.0    0.0               0.0           14.333333
4164.0 -8762.0    2001-06       1.0          1.0            1.0    0.0               0.0           16.666667
4164.0 -8762.0    2001-07       0.0          0.0            0.0    1.0               1.0           14.000000
4164.0 -8762.0    2001-08       1.0          1.0            0.0    0.0          

`features_gridmonth` berisi 226.784 baris x 34 kolom dengan **0 NaN** - artinya seluruh fitur lag/rolling/tetangga sudah terisi bersih (sel awal diisi 0, bukan dibiarkan kosong). Contoh baris memperlihatkan lag dan share bekerja benar (mis. `share_night_lag1` menyalin `share_night` bulan sebelumnya). Tabel ini rapi dan langsung siap dimodelkan.


## 10. Ringkasan
- features_incident: level insiden + sinyal waktu (is_night/is_weekend) + grid, dipakai pseudo-labeling.
- features_gridmonth: panel sel x bulan, 10 kolom basis + fitur kausal (spasial, siklikal, tren/recency,
  komposisi, spillover, share malam/akhir-pekan).
- Semua fitur prediktor bersifat lagged (kausal), aman untuk modeling di Hands-On 2.

In [11]:
print('DONE. Artefak di', OUT_DIR)
print('features_gridmonth kolom:', list(fgm.columns))

DONE. Artefak di /kaggle/working
features_gridmonth kolom: ['gi', 'gj', 'period_str', 'n_crimes', 'n_violent', 'n_property', 'n_arrest', 'n_domestic', 'n_types', 'arrest_rate', 'domestic_rate', 'share_night', 'share_weekend', 'glat', 'glon', 'year_idx', 'pmonth', 'pquarter', 'pmonth_sin', 'pmonth_cos', 'lag_1', 'lag_3', 'lag_12', 'roll3_mean', 'roll6_mean', 'roll12_mean', 'cell_hist_mean', 'trend_3_12', 'n_violent_lag1', 'arrest_rate_lag1', 'domestic_rate_lag1', 'neighbor_lag1_mean', 'share_night_lag1', 'share_weekend_lag1']
